In [1]:
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate

# Workflow described in https://python.langchain.com/docs/tutorials/graph/

In [2]:
## -----------------------run for single question ----------------------
df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
subfolder = 'BenchmarkDataset2025'
dataset = 'benchmark2025'
model_name = 'gpt_4o'
model=models[model_name]()
config = {'configurable': {'model': model, 'model_name':model_name, 'attempts':10},'recursion_limit':50} #10 iterations eval-coree and 1 for generator
df_results = df.copy()    
i = 3933
row = df_results.iloc[i]
# for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        # if pd.isna(row['generated_cypher']):
question = row['Question'] if dataset == 'sustain' else row['question']
print(i,question)

# Get credentials
required_cols = {'database_name', 'url', 'username', 'password'}
if required_cols.issubset(df_results.columns):
    database_url = row['url']
    database_name = row['database_name']
    database_user = row['username']
    database_password = row['password']
else:
    #its the sustain dataset
    database_url = os.getenv('database_url')
    database_name = os.getenv('database_name')
    database_user = os.getenv('database_user')
    database_password = os.getenv('database_password')
inputs = {
        'question':question,
        'database_url': database_url,
        'database_name':database_name,
        'database_user':database_user,
        'database_password':database_password
    }
result =app.invoke(inputs, config =config) #,debug=False
print('Generated Cypher:', result['cypher_statement'])
print('Output:', result['database_records'])
print('Path:', result['path'])
print('Answer:', result['answer'])


3933 List the first 3 actors who have worked with both Tom Hanks and Peter Cornwell.
--------------------------------------------------Cypher Generator: START

MATCH (tom:Person {name: "Tom Hanks"})-[:CAST_FOR]->(m:Movie)<-[:CAST_FOR]-(actor:Person),
      (peter:Person {name: "Peter Cornwell"})-[:CAST_FOR]->(m)
RETURN DISTINCT actor.name
LIMIT 3
-------------------------------Cypher Generator: END
Current frequency of nodes in the path: Counter({'cypher_generator': 1})
----------------------------------------------Cypher Evaluator:START 
-----------------------------------------Checking for syntax errors...
-----------------------------------------Checking for relationship direction errors...
-----------------------------------------Checking for additional errors with LLM...
LLM output for Cypher evaluation: errors=['Label (:Person {name: \\"Peter Cornwell\\"}) does not exist, did you mean another name? The schema does not list \'Peter Cornwell\' as an example name for the Person node

In [2]:
def evaluate_model_usage (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}_usage_eleftheria.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}_usage_eleftheria.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        # df_results['cypher_errors_history'] = None           # Use None so we can store a list
        # df_results['cypher_errors'] = None           # Use None so we can store a list
        # df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        # timing columns
        df_results['eval_time_s'] = None              # <-- total wall-clock time per query
        # tokens columns
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list
    config = {'configurable': {'model': model, 'model_name':model_name,'attempts':10},'recursion_limit':50} #10 iterations eval-coree and 1 for generator
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            if i>=1501 and i<2000:
                if i==1868:
                    continue
                question = row['Question'] if dataset == 'sustain' else row['question']
                print(i,question)
            
                # Get credentials
                required_cols = {'database_name', 'url', 'username', 'password'}
                if required_cols.issubset(df_results.columns):
                    database_url = row['url']
                    database_name = row['database_name']
                    database_user = row['username']
                    database_password = row['password']
                else:
                    #its the sustain dataset
                    database_url = os.getenv('database_url')
                    database_name = os.getenv('database_name')
                    database_user = os.getenv('database_user')
                    database_password = os.getenv('database_password')
                    # activate it for gemma 9b
                    # if database_name == 'offshoreleaks':
                    #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                    #     continue
                    # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
                
                # Run the graph and save outputs
                start = time.perf_counter() # start wall-clock timer
                result =app.invoke(inputs, config =config) #,debug=False
                elapsed = time.perf_counter() - start # elapsed wall-clock time
                df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                # df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                # df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                # df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                # if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                #     df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                # else:
                #     df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                print("elapsed time:", round(elapsed, 4))
                torch.cuda.empty_cache()#clear gpu cache
                

                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
            else:
                print(f"Skipping row {i}, already computed.")
            
    return df_results

In [3]:
model_name = 'gpt_4o' # 'qwen2_5_Coder_32b' #'llama3_2_3b'
df_results = evaluate_model_usage (model_name, models[model_name](), 'benchmark2025')#sustain

Starting evaluation for gpt_4o on benchmark2025 ...


  0%|          | 0/2128 [00:00<?, ?it/s]

Skipping row 0, already computed.
Skipping row 1, already computed.
Skipping row 2, already computed.
Skipping row 3, already computed.
Skipping row 4, already computed.
Skipping row 5, already computed.
Skipping row 6, already computed.
Skipping row 7, already computed.
Skipping row 8, already computed.
Skipping row 9, already computed.
Skipping row 10, already computed.
Skipping row 11, already computed.
Skipping row 12, already computed.
Skipping row 13, already computed.
Skipping row 14, already computed.
Skipping row 15, already computed.
Skipping row 16, already computed.
Skipping row 17, already computed.
Skipping row 18, already computed.
Skipping row 19, already computed.
Skipping row 20, already computed.
Skipping row 21, already computed.
Skipping row 22, already computed.
Skipping row 23, already computed.
Skipping row 24, already computed.
Skipping row 25, already computed.
Skipping row 26, already computed.
Skipping row 27, already computed.
Skipping row 28, already compu

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct